# Sudoku-12 : Resolution avec Z3 SMT Solver (C#)

**Navigation** : [<< Sudoku-11 Choco C#](Sudoku-11-Choco-Csharp.ipynb) | [Index](README.md) | [Sudoku-13 Symbolic Automata C# >>](Sudoku-13-SymbolicAutomata-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Comprendre les principes de base de Z3** - SMT solver, satisfiabilite, theories et contraintes
2. **Modeliser un probleme de contraintes avec Z3** - Representation des variables, construction des contraintes, resolution
3. **Comparaison des approches de resolution** - Entiers vs vecteurs de bits, API de substitution et tactiques

### Prerequis

- Avoir suivi le notebook **[Sudoku-10 OR-Tools](Sudoku-10-ORTools-Csharp.ipynb)** (recommande pour comprendre les concepts de contraintes)
- Notions de base en programmation logique et en raisonnement symbolique

### Duree estimee : ~40 minutes

### Voir aussi

- [Search-8-CSP-Advanced](../Search/Foundations/Search-8-CSP-Advanced.ipynb) - Contraintes globales avancees
- [SymbolicAI-Linq2Z3](../SymbolicAI/) - Serie complete sur l'IA symbolique avec Z3

## Introduction a Z3

Z3 est un SMT solver (Satisfiability Modulo Theories) qui peut etre utilise pour resoudre des problemes de logique du premier ordre. Il permet de verifier la satisfiabilite d'une formule logique sous certaines contraintes et est particulierement efficace pour resoudre des problemes impliquant des contraintes complexes, comme les puzzles de Sudoku.

Un SMT solver combine des techniques de resolution SAT (Satisfiability) avec des theories comme l'arithmetique, les tableaux, les bit-vectors, etc., permettant ainsi de resoudre des problemes logiques beaucoup plus riches.

**References :**
- [Exemples en C#](https://github.com/Z3Prover/z3/blob/master/examples/dotnet/Program.cs)
- [Programming Z3](https://theory.stanford.edu/~nikolaj/programmingz3.html)
- [Z3Py Guide Examples](https://ericpony.github.io/z3py-tutorial/guide-examples.htm) en Python

## Configuration de l'environnement

In [ ]:
#r "nuget: Microsoft.Z3"

Installing Packages Microsoft.Z3

## Importation des Classes de Base

Nous allons importer les classes de base définies dans le notebook précédent, fournissant notamment la représentation, le chargement et l'affichage de Sudokus, et l'infrastructure de résolution.


In [ ]:
#!import Sudoku-0-Environment-Csharp.ipynb

# Notebook 0: Classes de Base pour la Résolution de Sudoku

Ce notebook contient les classes de base nécessaires pour la manipulation et la résolution des grilles de Sudoku. Il sera importé dans les autres notebooks pour réutiliser ces classes.

## Importation des Bibliothèques Nécessaires

Nous commençons par importer les bibliothèques nécessaires.


Installing Packages Plotly.NET

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


## Définition de l'interface ISudokuSolver

Nous définissons ici l'interface ISudokuSolver qui sera implémentée par les différentes stratégies de résolution de Sudoku.


## Définition de la classe SudokuHelper

Nous ajoutons ici la classe SudokuHelper qui contient des méthodes utilitaires pour charger  des grilles de Sudoku et tester des solvers.

- `GetSudokus` : Renvoie des listes de Sudoku issues de fichiers de 3 difficultés différentes.
- `SolveSudoku` : effectue un test simple d'un solver sur un sudoku donné.
- `TestSolvers` : exécute les tests de performance sur plusieurs solveurs.
- `DisplayResults` : affiche les résultats des tests sous forme de graphiques.



### 3. Implémentation de base avec des entiers

Comme nous allons tester plusieurs stratégies de résolution, nous commencerons par implémenter un solver simple utilisant des entiers pour représenter les cellules du Sudoku et fournissant les méthodes pour construire les contraintes.


In [ ]:
// Importer les bibliothèques nécessaires
using System.Diagnostics;
using System;
using System.Collections;
using System.Collections.Generic;
using Microsoft.Z3;

// Classe pour résoudre le Sudoku en utilisant Z3 avec des entiers
public class Z3IntSolverSimple : ISudokuSolver
{
    public static Context ctx = new Context();
    public static BoolExpr _GenericContraints;
    public static IntExpr[][] CellVariables = new IntExpr[9][];
    public Z3IntSolverSimple()
    {
        // Initialiser les variables de cellule
        for (uint i = 0; i < 9; i++)
        {
            CellVariables[i] = new IntExpr[9];
            for (uint j = 0; j < 9; j++)
                CellVariables[i][j] = (IntExpr)ctx.MkConst(ctx.MkSymbol("x_" + (i + 1) + "_" + (j + 1)), ctx.IntSort);
        }
    }

    // Contraintes génériques pour tous les Sudokus, conservées en mémoire pour éviter de les recalculer
    public static BoolExpr GenericContraints
    {
        get
        {
            if (_GenericContraints == null)
            {
                _GenericContraints = GetGenericConstraints();
            }
            return _GenericContraints;
        }
    }
    
    // Générer les contraintes génériques
    public static BoolExpr GetGenericConstraints()
    {
        // Chaque cellule contient une valeur entre 1 et 9
        Expr[][] cells_c = new Expr[9][];
        for (uint i = 0; i < 9; i++)
        {
            cells_c[i] = new BoolExpr[9];
            for (uint j = 0; j < 9; j++)
                cells_c[i][j] = ctx.MkAnd(ctx.MkLe(ctx.MkInt(1), CellVariables[i][j]),
                                          ctx.MkLe(CellVariables[i][j], ctx.MkInt(9)));
        }
        // Chaque ligne contient des chiffres distincts
        BoolExpr[] rows_c = new BoolExpr[9];
        for (uint i = 0; i < 9; i++)
            rows_c[i] = ctx.MkDistinct(CellVariables[i]);
        // Chaque colonne contient des chiffres distincts
        BoolExpr[] cols_c = new BoolExpr[9];
        for (uint j = 0; j < 9; j++)
        {
            IntExpr[] column = new IntExpr[9];
            for (uint i = 0; i < 9; i++)
                column[i] = CellVariables[i][j];
            cols_c[j] = ctx.MkDistinct(column);
        }
        // Chaque carré 3x3 contient des chiffres distincts
        BoolExpr[][] sq_c = new BoolExpr[3][];
        for (uint i0 = 0; i0 < 3; i0++)
        {
            sq_c[i0] = new BoolExpr[3];
            for (uint j0 = 0; j0 < 3; j0++)
            {
                IntExpr[] square = new IntExpr[9];
                for (uint i = 0; i < 3; i++)
                    for (uint j = 0; j < 3; j++)
                        square[3 * i + j] = CellVariables[3 * i0 + i][3 * j0 + j];
                sq_c[i0][j0] = ctx.MkDistinct(square);
            }
        }
        BoolExpr sudoku_c = ctx.MkTrue();
        foreach (BoolExpr[] t in cells_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(rows_c), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(cols_c), sudoku_c);
        foreach (BoolExpr[] t in sq_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);
        return sudoku_c;
    }

    // Générer les contraintes spécifiques à une grille de Sudoku donnée
    public BoolExpr GetPuzzleConstraints(SudokuGrid grid)
    {
        BoolExpr instance_c = ctx.MkTrue();
        for (uint i = 0; i < 9; i++)
            for (uint j = 0; j < 9; j++)
                if (grid.Cells[i,j] != 0)
                {
                    instance_c = ctx.MkAnd(instance_c,
                        (BoolExpr)ctx.MkEq(CellVariables[i][j], ctx.MkInt(grid.Cells[i,j])));
                }
        return instance_c;
    }

    // Résoudre le Sudoku
    public SudokuGrid Solve(SudokuGrid grid)
    {
        SudokuGrid solution = new SudokuGrid();
        var sudoku_c = GenericContraints;
        var instance_c = GetPuzzleConstraints(grid);
        Solver s = ctx.MkSolver();
        s.Assert(sudoku_c);
        s.Assert(instance_c);        
        if (s.Check() == Status.SATISFIABLE)
        {
            Model m = s.Model;
            for (uint i = 0; i < 9; i++)
            {
                for (uint j = 0; j < 9; j++)
                {
                    solution.Cells[i,j] = ((IntNum)m.Evaluate(CellVariables[i][j])).Int;
                }
            }
        }
        else
        {
            Console.WriteLine("Failed to solve sudoku");
            throw new Exception("Failed to solve sudoku");
        }
        return solution;
    }
}

### 7. Utilisation de vecteurs de bits

Nous allons maintenant explorer une autre piste d'amélioration en utilisant des vecteurs de bits pour représenter les cellules du Sudoku. Cette approche peut réduire la taille des variables et améliorer les performances du solver.

Comme nous testerons plusieurs types de résolution, nous introduisons une classe de base qui fournit les contraintes.

In [ ]:
using Microsoft.Z3;

public abstract class Z3BitVectorSolverBase : ISudokuSolver
{
    public static Context ctx = new Context();
    public static BoolExpr _GenericContraints;
    public static BitVecExpr[][] CellVariables = new BitVecExpr[9][];

    private static Sort BitVectorSort = ctx.MkBitVecSort(4);

    public static Solver _ReusableSolver;

    public Z3BitVectorSolverBase()
    {
        // Initialiser les variables de cellule en tant que vecteurs de 4 bits
        for (uint i = 0; i < 9; i++)
        {
            CellVariables[i] = new BitVecExpr[9];
            for (uint j = 0; j < 9; j++)
                CellVariables[i][j] = (BitVecExpr)ctx.MkConst(ctx.MkSymbol("x_" + (i + 1) + "_" + (j + 1)), BitVectorSort);
        }
    }

    // Contraintes génériques pour le Sudoku
    public static BoolExpr GenericContraints
    {
        get
        {
            if (_GenericContraints == null)
            {
                _GenericContraints = GetGenericConstraints();
            }
            return _GenericContraints;
        }
    }

    public static Solver ReusableSolver
    {
        get
        {
            if (_ReusableSolver == null)
            {
                _ReusableSolver = MakeReusableSolver();
            }
            return _ReusableSolver;
        }
    }

    public static Solver MakeReusableSolver()
    {
        Solver s = ctx.MkSolver();
        s.Assert(GenericContraints);
        return s;
    }

    // Obtenir l'expression constante pour une valeur donnée
    protected static BitVecExpr GetConstExpr(int value)
    {
        return (BitVecExpr)ctx.MkNumeral(value, BitVectorSort);
    }

    // Générer les contraintes génériques pour les vecteurs de bits
    public static BoolExpr GetGenericConstraints()
    {
        // Chaque cellule contient une valeur entre 1 et 9
        Expr[][] cells_c = new Expr[9][];
        for (uint i = 0; i < 9; i++)
        {
            cells_c[i] = new BoolExpr[9];
            for (uint j = 0; j < 9; j++)
                cells_c[i][j] = ctx.MkAnd(ctx.MkBVULE(GetConstExpr(1), CellVariables[i][j]),
                    ctx.MkBVULE(CellVariables[i][j], GetConstExpr(9)));
        }

        // Chaque ligne contient des chiffres distincts
        BoolExpr[] rows_c = new BoolExpr[9];
        for (uint i = 0; i < 9; i++)
            rows_c[i] = ctx.MkDistinct(CellVariables[i]);

        // Chaque colonne contient des chiffres distincts
        BoolExpr[] cols_c = new BoolExpr[9];
        for (uint j = 0; j < 9; j++)
        {
            BitVecExpr[] column = new BitVecExpr[9];
            for (uint i = 0; i < 9; i++)
                column[i] = CellVariables[i][j];

            cols_c[j] = ctx.MkDistinct(column);
        }

        // Chaque carré 3x3 contient des chiffres distinct
        BoolExpr[][] sq_c = new BoolExpr[3][];
        for (uint i0 = 0; i0 < 3; i0++)
        {
            sq_c[i0] = new BoolExpr[3];
            for (uint j0 = 0; j0 < 3; j0++)
            {
                BitVecExpr[] square = new BitVecExpr[9];
                for (uint i = 0; i < 3; i++)
                for (uint j = 0; j < 3; j++)
                    square[3 * i + j] = CellVariables[3 * i0 + i][3 * j0 + j];
                sq_c[i0][j0] = ctx.MkDistinct(square);
            }
        }

        BoolExpr sudoku_c = ctx.MkTrue();
        foreach (BoolExpr[] t in cells_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(rows_c), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(cols_c), sudoku_c);
        foreach (BoolExpr[] t in sq_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);

        return sudoku_c;
    }

    // Générer les contraintes spécifiques à une grille de Sudoku donnée
    public BoolExpr GetPuzzleConstraints(SudokuGrid grid)
    {
        BoolExpr instance_c = ctx.MkTrue();
        for (uint i = 0; i < 9; i++)
        for (uint j = 0; j < 9; j++)
            if (grid.Cells[i,j] != 0)
            {
                instance_c = ctx.MkAnd(instance_c,
                    (BoolExpr)ctx.MkEq(CellVariables[i][j], GetConstExpr(grid.Cells[i,j])));
            }

        return instance_c;
    }

    // Méthode abstraite pour résoudre le Sudoku
    public abstract SudokuGrid Solve(SudokuGrid s);
}


### 4. Implémentation simple avec vecteurs de bits

Nous allons implémenter un solver simple en utilisant des vecteurs de bits.

In [ ]:
using Microsoft.Z3;

public class Z3BitVectorSolverSimple : Z3BitVectorSolverBase
{
    public override SudokuGrid Solve(SudokuGrid s)
    {
        SudokuGrid solution = new SudokuGrid();
        SudokuSolve(s, ref solution);
        return solution;
    }

    public void SudokuSolve(SudokuGrid grid, ref SudokuGrid solution)
    {
        var sudoku_c = GenericContraints;
        var instance_c = GetPuzzleConstraints(grid);
        Solver s = ctx.MkSolver();
        s.Assert(sudoku_c);
        s.Assert(instance_c);

        if (s.Check() == Status.SATISFIABLE)
        {
            Model m = s.Model;
            for (uint i = 0; i < 9; i++)
            {
                for (uint j = 0; j < 9; j++)
                {
                    solution.Cells[i,j] = ((BitVecNum)m.Evaluate(CellVariables[i][j])).Int;
                }
            }
        }
        else
        {
            Console.WriteLine("Failed to solve sudoku");
            throw new Exception("Failed to solve sudoku");
        }
    }
}


### 5. Utilisation de l'API de substitution avec vecteurs de bits

Nous allons implémenter une classe utilisant l'API de substitution. Cette approche peut améliorer les performances en réutilisant les contraintes génériques et en substituant uniquement les valeurs spécifiques à la grille de Sudoku en cours de résolution.


In [ ]:
using Microsoft.Z3;


public class Z3BitVectorSolverSubstitution : Z3BitVectorSolverBase
{
    public override SudokuGrid Solve(SudokuGrid s)
    {
        SudokuGrid solution = new SudokuGrid();
        SudokuSolve(s, ref solution);
        return solution;
    }

    public void SudokuSolve(SudokuGrid grid, ref SudokuGrid solution)
    {
        var substExprs = new List<Expr>();
        var substVals = new List<Expr>();

        for (int i = 0; i < 9; i++)
        for (int j = 0; j < 9; j++)
            if (grid.Cells[i,j] != 0)
            {
                substExprs.Add(CellVariables[i][j]);
                substVals.Add(GetConstExpr(grid.Cells[i,j]));
            }

        // Utiliser l'API de substitution pour appliquer les contraintes spécifiques
        BoolExpr instance_c = (BoolExpr)GenericContraints.Substitute(substExprs.ToArray(), substVals.ToArray());
        Solver solver = ctx.MkSolver();
        solver.Assert(instance_c);
        if (solver.Check() == Status.SATISFIABLE)
        {
            Model m = solver.Model;
            for (uint i = 0; i < 9; i++)
            {
                for (uint j = 0; j < 9; j++)
                {
                    if (grid.Cells[i,j] == 0)
                    {
                        solution.Cells[i,j] = ((BitVecNum)m.Evaluate(CellVariables[i][j])).Int;
                    }
                    else
                    {
                        solution.Cells[i,j] = grid.Cells[i,j];
                    }
                }
            }
        }
        else
        {
            Console.WriteLine("Failed to solve sudoku");
            throw new Exception("Failed to solve sudoku");
        }
    }
}


### 6. Utilisation de l'API de tactiques avec vecteurs de bits

Nous allons maintenant explorer l'utilisation de l'API de tactiques de Z3 pour résoudre les Sudokus. Les tactiques permettent d'appliquer des transformations et des simplifications spécifiques pour améliorer l'efficacité de la résolution.

In [ ]:
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;

internal class Z3BitSub_Tactic : Z3BitVectorSolverBase
{
    public override SudokuGrid Solve(SudokuGrid s)
    {
        SudokuGrid solution = new SudokuGrid();
        SudokuSolve(s, ref solution);
        return solution;
    }
    public void SudokuSolve(SudokuGrid grid, ref SudokuGrid solution)
    {
        var substExprs = new List<Expr>();
        var substVals = new List<Expr>();
        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
                if (grid.Cells[i,j] != 0)
                {
                    substExprs.Add(CellVariables[i][j]);
                    substVals.Add(GetConstExpr(grid.Cells[i,j]));
                }
        BoolExpr instance_c = (BoolExpr)GenericContraints.Substitute(substExprs.ToArray(), substVals.ToArray());
        Solver solver = ctx.MkSolver();
        solver.Assert(instance_c);
        BoolExpr puzzleConstraints = GetPuzzleConstraints(grid);
        
        // Utiliser l'API de tactiques pour simplifier et résoudre le Sudoku
        Tactic tactic = ctx.MkTactic("simplify");
        Goal goal = ctx.MkGoal();
        goal.Assert(ctx.MkAnd(GenericContraints, puzzleConstraints));
        ApplyResult applyResult = tactic.Apply(goal);
        if (applyResult.NumSubgoals > 0)
        {
            Goal newGoal = applyResult.Subgoals[0];
            solver.Assert(newGoal.Formulas);
            if (solver.Check() == Status.SATISFIABLE)
            {
                Model m = solver.Model;
                for (uint i = 0; i < 9; i++)
                {
                    for (uint j = 0; j < 9; j++)
                    {
                        if (grid.Cells[i,j] == 0)
                        {
                            solution.Cells[i,j] = ((BitVecNum)m.Evaluate(CellVariables[i][j])).Int;
                        }
                        else
                        {
                            solution.Cells[i,j] = grid.Cells[i,j];
                        }
                    }
                }
            }
            else
            {
                Console.WriteLine("Failed to solve sudoku");
                throw new Exception("Failed to solve sudoku");
            }
        }
    }
}


### 7. Comparaison des solveurs

Pour comparer les différentes implémentations de solveurs, nous utiliserons une approche similaire à celle de notre notebook OR-Tools. Nous évaluerons les performances de chaque solver sur des puzzles de Sudoku de différentes difficultés (Facile, Moyen, Difficile).

In [ ]:
var solvers = new List<(string Name, ISudokuSolver Solver)>
{
    ("Z3 Int Solver Simple", new Z3IntSolverSimple()),
    ("Z3 Bit Vector Solver Simple", new Z3BitVectorSolverSimple()),
    ("Z3 Bit Vector Solver Substitution", new Z3BitVectorSolverSubstitution()),
    ("Z3 Bit Vector Solver Tactic", new Z3BitSub_Tactic())
};

var results = SudokuHelper.TestSolvers(solvers);
SudokuHelper.DisplayResults(results);



Running tests...

### Interpretation des resultats

Le tableau ci-dessus presente les performances des differentes implementations de solveurs Z3 sur les trois niveaux de difficulte.

| Solveur | Approche | Avantages | Inconvenients |
|---------|----------|-----------|---------------|
| Z3 Int Solver Simple | Entiers | Implementation directe, facile a comprendre | Variables plus lourdes (32 bits) |
| Z3 Bit Vector Simple | Vecteurs 4 bits | Meilleure performance pour les valeurs petites | Meme approche que l'int solver |
| Z3 Bit Vector Substitution | Substitution | Reutilise les contraintes generiques | Plus complexe a mettre en oeuvre |
| Z3 Bit Vector Tactic | Tactiques | Simplification automatique | Overhead potentiel |

**Points cles** :

1. **Vecteurs de bits** : L'utilisation de vecteurs de 4 bits est optimale pour les Sudoku (valeurs 1-9), reduisant la taille des variables par rapport aux entiers 32 bits.

2. **API de substitution** : Permet de reutiliser les contraintes generiques en substituant uniquement les valeurs specifiques, evitant de redeclarer toutes les contraintes.

3. **Tactiques** : Les tactiques comme "simplify" permettent de pre-traiter les contraintes pour optimiser la resolution, mais l'overhead peut ne pas etre justifie pour des Sudoku simples.

4. **Performance** : Pour les Sudoku, tous les solveurs sont performants. Les differences deviennent plus significatives sur des problemes de contraintes plus complexes.

> **Note technique** : Les solveurs Z3 utilisent un contexte statique partage pour optimiser la performance. Dans une application de production, il faudrait gerer le cycle de vie du contexte plus proprement.


### Conclusion

Dans ce notebook, nous avons exploré différentes approches pour résoudre des puzzles de Sudoku en utilisant Z3. Nous avons commencé par une implémentation simple en utilisant des entiers, puis nous avons introduit des méthodes plus sophistiquées utilisant des vecteurs de bits et l'API de substitution. Nous avons également comparé les performances de ces différentes approches.

Les résultats montrent que les solveurs utilisant des vecteurs de bits et l'API de substitution offrent des performances améliorées par rapport à l'approche simple basée sur les entiers. L'utilisation des tactiques de simplification peut également apporter des gains de performance significatifs mais nécessite plus de tests.

Merci d'avoir suivi ce notebook, et j'espère que cela vous a permis de mieux comprendre comment utiliser Z3 pour résoudre des problèmes de contraintes complexes comme les puzzles de Sudoku.

## Exercices

### Exercice 1 : Implémenter le Sudoku X avec diagonales

Le **Sudoku X** est une variante ou les deux diagonales principales doivent egalement contenir des chiffres distincts.

**Objectif** : Etendre `Z3BitVectorSolverBase` pour ajouter des contraintes de diagonale.

**Indices** :
1. La diagonale principale contient `CellVariables[i][i]` pour i de 0 a 8
2. L'anti-diagonale contient `CellVariables[i][8-i]` pour i de 0 a 8
3. Utiliser `ctx.MkDistinct()` comme pour les lignes et colonnes
4. `ctx` et `CellVariables` sont accessibles statiquement depuis la classe de base

**Verification** : Verifiez d'abord avec `EnforceDiagonals = false` (comportement identique au solveur de base).

In [ ]:
// Exercice 1 : Sudoku X avec contraintes de diagonale (Z3 BitVector)
using Microsoft.Z3;

public class SudokuXZ3Solver : Z3BitVectorSolverBase
{
    public bool EnforceDiagonals { get; set; } = true;

    public override SudokuGrid Solve(SudokuGrid grid)
    {
        var solver = ctx.MkSolver();

        // TODO : Ajouter les contraintes generiques standards
        // solver.Assert(GenericContraints);

        if (EnforceDiagonals)
        {
            // TODO : Extraire et contraindre la diagonale principale
            // var mainDiag = new BitVecExpr[9];
            // for (int i = 0; i < 9; i++) mainDiag[i] = CellVariables[i][i];
            // solver.Assert(ctx.MkDistinct(mainDiag));

            // TODO : Extraire et contraindre l'anti-diagonale
            // var antiDiag = new BitVecExpr[9];
            // for (int i = 0; i < 9; i++) antiDiag[i] = CellVariables[i][8 - i];
            // solver.Assert(ctx.MkDistinct(antiDiag));
        }

        // TODO : Ajouter les contraintes du puzzle et resoudre
        // solver.Assert(GetPuzzleConstraints(grid));
        // if (solver.Check() == Status.SATISFIABLE) { ... extraire la solution ... }

        throw new NotImplementedException("A vous d'implementer le Sudoku X avec Z3 !");
    }
}

Console.WriteLine("TODO : Implementez SudokuXZ3Solver avec les contraintes de diagonale");

### Exercice 2 : Verifier l'unicite d'une solution avec Z3

Z3 permet de chercher toutes les solutions en ajoutant progressivement des contraintes d'exclusion.

**Objectif** : Implementer `HasUniqueSolution` qui verifie qu'un puzzle a exactement une solution.

**Indices** :
1. Trouver la premiere solution S1
2. Construire la contrainte d'exclusion : "au moins une cellule differe de S1"
   - `ctx.MkOr(ctx.MkNot(ctx.MkEq(CellVariables[i][j], valeur_S1)))` pour chaque cellule
3. Si `solver.Check()` retourne `UNSATISFIABLE` apres ajout : la solution est unique
4. Technique appelee "exclusion de modele" (model blocking)

**Verification** : Testez sur un puzzle facile (solution unique attendue).

In [ ]:
// Exercice 2 : Verification d'unicite des solutions avec Z3
public class Z3SudokuUnicityChecker
{
    private static Context ctx = Z3IntSolverSimple.ctx;
    private static IntExpr[][] CellVariables = Z3IntSolverSimple.CellVariables;

    /// <summary>
    /// Verifie si un puzzle a exactement une solution.
    /// Strategie : trouver S1, puis chercher S2 != S1. Si impossible -> unique.
    /// </summary>
    public bool HasUniqueSolution(SudokuGrid puzzle)
    {
        var s = ctx.MkSolver();

        // TODO : 1. Ajouter les contraintes generiques et celles du puzzle
        // s.Assert(Z3IntSolverSimple.GenericContraints);
        // Indice : voir Z3IntSolverSimple.GetPuzzleConstraints(puzzle)

        // TODO : 2. Verifier satisfiabilite et extraire S1
        // if (s.Check() != Status.SATISFIABLE) return false;
        // Model m1 = s.Model;

        // TODO : 3. Construire la contrainte "au moins une cellule differe de S1"
        // BoolExpr[] differences = new BoolExpr[81];
        // for (int i = 0; i < 9; i++)
        //     for (int j = 0; j < 9; j++)
        //     {
        //         var val = ((IntNum)m1.Evaluate(CellVariables[i][j])).Int;
        //         differences[i * 9 + j] = ctx.MkNot(ctx.MkEq(CellVariables[i][j], ctx.MkInt(val)));
        //     }
        // s.Assert(ctx.MkOr(differences));

        // TODO : 4. Retourner true si aucune deuxieme solution n'existe
        // return s.Check() == Status.UNSATISFIABLE;

        throw new NotImplementedException("A vous d'implementer HasUniqueSolution !");
    }
}

// Test
var checker2 = new Z3SudokuUnicityChecker();
var easyPuzzle2 = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
// bool unique = checker2.HasUniqueSolution(easyPuzzle2);
// Console.WriteLine($"Solution unique : {unique}");
Console.WriteLine("TODO : Implementez Z3SudokuUnicityChecker.HasUniqueSolution()");

### Exercice 3 : Comparer les tactiques Z3

Z3 propose differentes tactiques pour pre-traiter les formules avant resolution. Chaque tactique a ses avantages selon le type de probleme.

**Objectif** : Mesurer l'impact des tactiques sur les performances de resolution.

**Indices** :
1. Creer une tactique : `ctx.MkTactic("simplify")` ou `ctx.MkTactic("bit-blast")`
2. Creer un goal : `ctx.MkGoal()` puis `goal.Assert(contraintes)`
3. Appliquer : `tactic.Apply(goal)` retourne un `ApplyResult`
4. Passer les sous-goals au solveur : `solver.Assert(result.Subgoals[0].Formulas)`
5. Tactiques a tester : `"simplify"`, `"bit-blast"`, `"ctx-solver-simplify"`, `"solve-eqs"`

**Verification** : `"bit-blast"` est generalement efficace pour les BitVectors.

In [ ]:
// Exercice 3 : Comparaison des tactiques Z3
using Microsoft.Z3;
using System.Diagnostics;

var tacticPuzzle = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).First();
var tacticCtx = Z3BitVectorSolverBase.ctx;
var genericConstr = Z3BitVectorSolverBase.GenericContraints;
var puzzleConstrHelper = new Z3BitVectorSolverSimple();
var puzzleConstr = puzzleConstrHelper.GetPuzzleConstraints(tacticPuzzle);

string[] tacticNames = { "simplify", "ctx-solver-simplify", "bit-blast", "solve-eqs" };

Console.WriteLine("Comparaison des tactiques Z3 :");
Console.WriteLine(new string('-', 40));

// TODO : Pour chaque tactique, creer et appliquer, mesurer le temps de resolution
// foreach (var tacticName in tacticNames)
// {
//     var tactic = tacticCtx.MkTactic(tacticName);
//     var goal = tacticCtx.MkGoal();
//     goal.Assert(tacticCtx.MkAnd(genericConstr, puzzleConstr));
//     var sw = Stopwatch.StartNew();
//     var result = tactic.Apply(goal);
//     var solver = tacticCtx.MkSolver();
//     if (result.NumSubgoals > 0)
//         solver.Assert(result.Subgoals[0].Formulas);
//     var status = solver.Check();
//     sw.Stop();
//     Console.WriteLine($"{tacticName,-25} : {sw.ElapsedMilliseconds,6} ms - {status}");
// }

Console.WriteLine("TODO : Implementez la comparaison des tactiques Z3");

## Exercice : Verification de proprietes avec Z3

### Enonce

Utilisez Z3 pour verifier des proprietes logiques sur les grilles de Sudoku. Implementez les fonctions suivantes avec l'API Z3 en C# :

1. `HasUniqueSolution(SudokuGrid puzzle)` : Verifie si une grille a exactement une solution
   - Trouver la premiere solution S1
   - Ajouter la contrainte "la solution n'est pas S1"
   - Si aucune deuxieme solution -> unicite verifiee
   
2. `IsMinimal(SudokuGrid puzzle)` : Verifie si le puzzle est minimal (supprimer une valeur initiale cree une ambiguite)
   - Pour chaque cellule fixe (r, c) avec valeur v :
     - Creer une grille sans cette cellule
     - Verifier que `HasUniqueSolution` retourne `false` (plus d'une solution)

3. Testez avec le puzzle facile : verifiez unicite et minimalite

### Indice

Pour `HasUniqueSolution`, apres avoir trouve S1 = {cells[i][j] = v[i][j]}, ajoutez la contrainte `Or(cell[i][j] != v[i][j])` pour toutes les cellules. Cette contrainte dit "au moins une cellule differe de S1".

In [ ]:
// EXERCICE : Verification de proprietes avec Z3

public class Z3SudokuPropertyChecker
{
    private static Context ctx = Z3IntSolverSimple.ctx;

    /// <summary>
    /// Verifie si un puzzle a exactement une solution.
    /// Strategie : trouver S1, puis chercher S2 != S1. Si impossible -> unique.
    /// </summary>
    public bool HasUniqueSolution(SudokuGrid puzzle)
    {
        // TODO : Implementer la verification d'unicite
        // 1. Construire les contraintes generiques + contraintes du puzzle
        // 2. Trouver la premiere solution S1
        // 3. Ajouter la contrainte "au moins une cellule differe de S1"
        // 4. Verifier si une deuxieme solution existe
        throw new NotImplementedException("A vous de jouer !");
    }

    /// <summary>
    /// Verifie si le puzzle est minimal :
    /// supprimer n'importe quelle valeur initiale cree une ambiguite.
    /// </summary>
    public bool IsMinimal(SudokuGrid puzzle)
    {
        // TODO : Implementer la verification de minimalite
        // Pour chaque cellule fixee (i, j) :
        //   - Creer une copie du puzzle sans la valeur en (i, j)
        //   - Verifier que HasUniqueSolution retourne false
        // Si toutes les cellules satisfont cette condition -> minimal
        throw new NotImplementedException("A vous de jouer !");
    }
}

// Test : verifier les proprietes sur le puzzle facile
var checker = new Z3SudokuPropertyChecker();
var easyPuzzle = SudokuHelper.GetHardestPuzzles(1)[0]; // Remplacer par un puzzle facile si disponible
Console.WriteLine("Test de verification de proprietes Z3");
Console.WriteLine("Puzzle selectionne :");
Console.WriteLine(easyPuzzle);
// bool unique = checker.HasUniqueSolution(easyPuzzle);
// bool minimal = checker.IsMinimal(easyPuzzle);
// Console.WriteLine($"Solution unique : {unique}");
// Console.WriteLine($"Puzzle minimal : {minimal}");

---

**Navigation** : [<< Sudoku-11 Choco C#](Sudoku-11-Choco-Csharp.ipynb) | [Index](README.md) | [Sudoku-13 Symbolic Automata C# >>](Sudoku-13-SymbolicAutomata-Csharp.ipynb)

**Voir aussi** : 
- [Search-8-CSP-Advanced](../Search/Foundations/Search-8-CSP-Advanced.ipynb) - Contraintes globales avancees
- [SymbolicAI](../SymbolicAI/README.md) - Serie sur l'IA symbolique et Z3